# 📐 PCB Layout Experiments Notebook

Explore placement strategies, constraints, and scoring.

Goals:
- Compare placement strategies
- Apply constraints (power, thermal, grouping)
- Visualize layouts
- Evaluate layout quality

In [ ]:
# Setup
from generation.layout.auto_place import auto_place
from generation.layout.constraints import apply_constraints
from generation.render.pcb_plot import draw_pcb

import copy
import pprint

## 🧩 Sample Design

In [ ]:
design = {
    "components": [
        {"ref": "U1", "value": "MCU"},
        {"ref": "R1", "value": "10k"},
        {"ref": "R2", "value": "1k"},
        {"ref": "C1", "value": "100nF"},
        {"ref": "U2", "value": "7805"},
    ],
    "nets": [
        {"name": "VCC", "connections": ["U2:OUT", "U1:VCC"]},
        {"name": "GND", "connections": ["U1:GND", "C1:2"]},
        {"name": "SIG", "connections": ["U1:PB0", "R1:1"]},
    ]
}

## 🧩 Baseline Placement

In [ ]:
baseline = auto_place(copy.deepcopy(design))

pprint.pprint(baseline["layout"])
draw_pcb(baseline)

## ⚡ Apply Power + Thermal Constraints

In [ ]:
constraints = {
    "power_zone": True,
    "thermal": True
}

power_layout = apply_constraints(copy.deepcopy(baseline), constraints)

draw_pcb(power_layout)

## 🔗 Apply Grouping (Net-Based)

In [ ]:
group_layout = apply_constraints(copy.deepcopy(baseline), {
    "grouping": True
})

draw_pcb(group_layout)

## 📏 Layout Scoring Function

In [ ]:
def layout_score(layout):
    """
    Lower is better
    """
    score = 0
    for pos in layout.values():
        score += abs(pos["x"]) + abs(pos["y"])
    return score

## 📊 Compare Strategies

In [ ]:
print("Baseline Score:", layout_score(baseline["layout"]))
print("Power Score:", layout_score(power_layout["layout"]))
print("Grouping Score:", layout_score(group_layout["layout"]))

## 🔁 Iterative Optimization (Search)

In [ ]:
best = None
best_score = float("inf")

for i in range(10):
    candidate = auto_place(copy.deepcopy(design))
    candidate = apply_constraints(candidate, {"grouping": True})

    score = layout_score(candidate["layout"])

    if score < best_score:
        best = candidate
        best_score = score

print("Best Score:", best_score)
draw_pcb(best)

## 🚀 Next Experiments

- Add ML-based placement
- Add RL-based optimization
- Add congestion-aware placement
- Integrate routing feedback loop